In [127]:
import pandas as pd

py_df = pd.read_csv('data/py_repos.csv', usecols=['rank', 'username/repo_name'])
all_safetensor_keys = set(f'{row["rank"]}.{row["username/repo_name"]}' for _, row in py_df.iterrows())
all_safetensor_keys

{'100.chubin/wttr.in',
 '11.josephmisiti/awesome-machine-learning',
 '12.public-apis/public-apis',
 '13.donnemartin/system-design-primer',
 '13.huggingface/transformers',
 '15.psf/requests',
 '16.521xueweihan/HelloGitHub',
 '18.scrapy/scrapy',
 '19.soimort/you-get',
 '20.minimaxir/big-list-of-naughty-strings',
 '21.ageitgey/face_recognition',
 '22.apache/superset',
 '23.TheAlgorithms/Python',
 '24.deepfakes/faceswap',
 '25.3b1b/manim',
 '25.jackfrued/Python-100-Days',
 '26.tiangolo/fastapi',
 '26.vinta/awesome-python',
 '27.ytdl-org/youtube-dl',
 '28.fighting41love/funNLP',
 '29.0voice/interview_internal_reference',
 '30.localstack/localstack',
 '31.isocpp/CppCoreGuidelines',
 '32.apachecn/AiLearning',
 '33.pandas-dev/pandas',
 '34.floodsung/Deep-Learning-Papers-Reading-Roadmap',
 '35.testerSunshine/12306',
 '36.faif/python-patterns',
 '37.google-research/bert',
 '38.getsentry/sentry',
 '39.CorentinJ/Real-Time-Voice-Cloning',
 '40.swisskyrepo/PayloadsAllTheThings',
 '41.willmcgugan/ric

In [124]:
from safetensors.torch import load_file
import polars as pl
from pathlib import Path
import pandas as pd
# defaultdict
from collections import defaultdict
import torch


schema = [('layer', pl.Int8), ('expert', pl.Int8)]
for _, row in py_df.iterrows():
    repo = row["username/repo_name"]
    schema.append((f'{repo}.w1', pl.Float16))
    schema.append((f'{repo}.w3', pl.Float16))

df = pl.DataFrame(schema=schema)

for layer in range(32):
    layer_path = Path(f'output/py/layer={layer:02d}')
    for expert in range(8):
        repos_to_activations = {}
        expert_path = layer_path / f'expert={expert}'

        w1_path = expert_path / 'w=1.safetensors'
        w3_path = expert_path / 'w=3.safetensors'

        w1 = load_file(w1_path)
        w3 = load_file(w3_path)

        assert set(w1.keys()) == all_safetensor_keys, f"Repo missing in w=1 for expert {expert}"
        assert set(w3.keys()) == all_safetensor_keys, f"Repo missing in w=3 for expert {expert}"

        for _, row in py_df.iterrows():
            key = f'{row["rank"]}.{row["username/repo_name"]}'

            w1_activation = w1[key].cpu().to(torch.float16).numpy()
            assert w1_activation.shape == (14336,), f"Unexpected shape for {key} in w=1: {w1_activation.shape}"
            repos_to_activations[f'{row["username/repo_name"]}.w1'] = w1_activation


            w3_activation = w3[key].cpu().to(torch.float16).numpy()
            assert w3_activation.shape == (14336,), f"Unexpected shape for {key} in w=3: {w3_activation.shape}"
            repos_to_activations[f'{row["username/repo_name"]}.w3'] = w3_activation
        
        curr_expert_df = pl.DataFrame(repos_to_activations)
        curr_expert_df = curr_expert_df.with_columns(pl.lit(layer, dtype=pl.Int8).alias("layer"), pl.lit(expert, dtype=pl.Int8).alias("expert"))
        curr_expert_df = curr_expert_df.select(pl.col(df.columns))


        df = pl.concat([df, curr_expert_df], how="vertical")

In [126]:
df

layer,expert,public-apis/public-apis.w1,public-apis/public-apis.w3,donnemartin/system-design-primer.w1,donnemartin/system-design-primer.w3,TheAlgorithms/Python.w1,TheAlgorithms/Python.w3,jackfrued/Python-100-Days.w1,jackfrued/Python-100-Days.w3,vinta/awesome-python.w1,vinta/awesome-python.w3,ytdl-org/youtube-dl.w1,ytdl-org/youtube-dl.w3,tensorflow/models.w1,tensorflow/models.w3,nvbn/thefuck.w1,nvbn/thefuck.w3,django/django.w1,django/django.w3,pallets/flask.w1,pallets/flask.w3,keras-team/keras.w1,keras-team/keras.w3,httpie/httpie.w1,httpie/httpie.w3,scikit-learn/scikit-learn.w1,scikit-learn/scikit-learn.w3,ansible/ansible.w1,ansible/ansible.w3,shadowsocks/shadowsocks.w1,shadowsocks/shadowsocks.w3,python/cpython.w1,python/cpython.w3,home-assistant/core.w1,home-assistant/core.w3,josephmisiti/awesome-machine-learning.w1,…,magenta/magenta.w3,locustio/locust.w1,locustio/locust.w3,pytorch/examples.w1,pytorch/examples.w3,jumpserver/jumpserver.w1,jumpserver/jumpserver.w3,luong-komorebi/Awesome-Linux-Software.w1,luong-komorebi/Awesome-Linux-Software.w3,PaddlePaddle/Paddle.w1,PaddlePaddle/Paddle.w3,reddit-archive/reddit.w1,reddit-archive/reddit.w3,charlax/professional-programming.w1,charlax/professional-programming.w3,instillai/TensorFlow-Course.w1,instillai/TensorFlow-Course.w3,python-poetry/poetry.w1,python-poetry/poetry.w3,open-mmlab/mmdetection.w1,open-mmlab/mmdetection.w3,toml-lang/toml.w1,toml-lang/toml.w3,junyanz/pytorch-CycleGAN-and-pix2pix.w1,junyanz/pytorch-CycleGAN-and-pix2pix.w3,bokeh/bokeh.w1,bokeh/bokeh.w3,sanic-org/sanic.w1,sanic-org/sanic.w3,binux/pyspider.w1,binux/pyspider.w3,nginx-proxy/nginx-proxy.w1,nginx-proxy/nginx-proxy.w3,cookiecutter/cookiecutter.w1,cookiecutter/cookiecutter.w3,chubin/wttr.in.w1,chubin/wttr.in.w3
i8,i8,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,…,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16
0,0,-0.027466,0.024536,0.0546875,0.271484,0.133789,-0.088867,-0.041992,-0.085938,0.004669,0.00705,-0.043701,0.030762,-0.141602,0.084473,0.036621,0.019775,0.031494,0.122559,-0.023438,-0.160156,-0.035156,-0.054688,-0.052979,-0.090332,-0.039062,-0.100098,0.023071,0.205078,0.017212,0.163086,0.209961,0.069824,-0.048584,-0.103516,0.158203,…,0.091797,-0.020508,0.007751,0.011475,-0.170898,-0.034424,0.0859375,0.019531,-0.019287,0.080078,-0.025879,-0.061523,0.143555,-0.012329,-0.031494,0.011353,0.012817,-0.040283,0.128906,0.004944,0.057617,-0.128906,-0.332031,0.111816,0.036133,-0.12793,-0.188477,-0.080078,0.05542,-0.034912,-0.155273,0.163086,0.069824,-0.388672,0.064453,-0.026367,0.038086
0,0,-0.130859,-0.15918,0.237305,-0.031494,-0.034668,0.10498,-0.084961,-0.010437,-0.003006,0.083984,-0.098145,0.119141,0.146484,-0.059082,-0.087891,-0.062012,-0.015137,-0.051758,0.029785,-0.065918,-0.046387,-0.032227,-0.146484,-0.071289,-0.015747,0.092773,0.106934,0.048584,0.004761,-0.320312,0.101074,0.09082,-0.118652,0.0859375,-0.162109,…,-0.006958,-0.398438,-0.014771,-0.128906,-0.033691,-0.261719,-0.053223,-0.310547,0.039307,-0.088379,-0.047852,0.076172,-0.057129,0.120117,0.447266,-0.033203,-0.125977,-0.101562,-0.089355,0.104004,0.037842,0.056885,0.084473,-0.059326,-0.21582,-0.233398,0.2578125,0.294922,0.118164,-0.02832,-0.131836,-0.054688,-0.163086,-0.259766,0.095215,0.080566,0.036865
0,0,0.189453,0.047852,0.060059,-0.000851,0.103027,0.019897,-0.169922,0.080078,0.109375,0.097656,0.036865,-0.061035,-0.161133,0.07959,0.267578,0.00824,-0.042725,-0.075195,0.155273,-0.241211,0.154297,-0.224609,0.062256,-0.040771,0.124023,0.049805,0.145508,-0.143555,0.111816,-0.151367,0.124512,0.02063,-0.152344,0.025757,0.093262,…,-0.015198,0.115723,-0.255859,0.058838,-0.188477,0.025391,-0.101074,-0.084473,0.033447,0.1484375,-0.010071,-0.034424,0.033691,0.046143,-0.010925,-0.139648,-0.069336,0.015564,-0.058838,-0.090332,0.086426,-0.182617,-0.048584,-0.036133,0.

In [99]:
schema = [(row["username/repo_name"], pl.List(pl.Float16)) for _, row in py_df.iterrows()]
df = pl.DataFrame(schema=schema)

public-apis/public-apis,donnemartin/system-design-primer,TheAlgorithms/Python,jackfrued/Python-100-Days,vinta/awesome-python,ytdl-org/youtube-dl,tensorflow/models,nvbn/thefuck,django/django,pallets/flask,keras-team/keras,httpie/httpie,scikit-learn/scikit-learn,ansible/ansible,shadowsocks/shadowsocks,python/cpython,home-assistant/core,josephmisiti/awesome-machine-learning,huggingface/transformers,psf/requests,521xueweihan/HelloGitHub,scrapy/scrapy,soimort/you-get,minimaxir/big-list-of-naughty-strings,ageitgey/face_recognition,apache/superset,deepfakes/faceswap,3b1b/manim,tiangolo/fastapi,fighting41love/funNLP,0voice/interview_internal_reference,localstack/localstack,isocpp/CppCoreGuidelines,apachecn/AiLearning,pandas-dev/pandas,floodsung/Deep-Learning-Papers-Reading-Roadmap,testerSunshine/12306,…,tornadoweb/tornado,eriklindernoren/ML-From-Scratch,google/python-fire,beurtschipper/Depix,keon/algorithms,wangzheng0822/algo,getredash/redash,tqdm/tqdm,nicolargo/glances,sebastianruder/NLP-progress,drduh/macOS-Security-and-Privacy-Guide,numpy/numpy,celery/celery,OWASP/CheatSheetSeries,facebookresearch/detectron2,microsoft/cascadia-code,deezer/spleeter,bitcoinbook/bitcoinbook,magenta/magenta,locustio/locust,pytorch/examples,jumpserver/jumpserver,luong-komorebi/Awesome-Linux-Software,PaddlePaddle/Paddle,reddit-archive/reddit,charlax/professional-programming,instillai/TensorFlow-Course,python-poetry/poetry,open-mmlab/mmdetection,toml-lang/toml,junyanz/pytorch-CycleGAN-and-pix2pix,bokeh/bokeh,sanic-org/sanic,binux/pyspider,nginx-proxy/nginx-proxy,cookiecutter/cookiecutter,chubin/wttr.in
list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],…,list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16]
